In [1]:
!pip install anndata tqdm transformers torch numpy pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.6/319.6 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 137.8 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import os

In [10]:
print(os.listdir('/content/drive/MyDrive/multiomics-project/data/processed'))


['token_dictionary.pkl', 'gene_median_dictionary.pkl', 'probe_ids_type3.csv', 'tcga_rna_seq.h5ad', 'tcga_methylation.h5ad', 'geneformer_embeddings.npy', 'geneformer_labels.npy']


 ### Set file paths

In [13]:
import os

BASE = "/content/drive/MyDrive/multiomics-project"

INPUT_H5AD        = f"{BASE}/data/processed/tcga_methylation.h5ad"
CPG_LIST_PATH     = f"{BASE}/data/processed/probe_ids_type3.csv"
OUTPUT_EMBEDDINGS = f"{BASE}/data/processed/methylgpt_embeddings.npy"

MODEL_DIR     = f"{BASE}/pretrained_models/methylgpt-medium"
MODEL_WEIGHTS = f"{MODEL_DIR}/small-best_model_epoch6.pt"
VOCAB_PATH    = f"{MODEL_DIR}/vocab.json"
ARGS_PATH     = f"{MODEL_DIR}/args.json"

In [14]:
print("Input exists:      ", os.path.exists(INPUT_H5AD))
print("Probe list exists: ", os.path.exists(CPG_LIST_PATH))
print("Model weights:     ", os.path.exists(MODEL_WEIGHTS))
print("Vocab:             ", os.path.exists(VOCAB_PATH))
print("Args:              ", os.path.exists(ARGS_PATH))

Input exists:       True
Probe list exists:  True
Model weights:      True
Vocab:              True
Args:               True


### Inspect the model config


In [15]:
import json
with open(ARGS_PATH) as f:
    args = json.load(f)
print(json.dumps(args, indent=2))

{
  "seed": 42,
  "input_type": "CpGs_type3",
  "parquet_dir": "processed_dataset/processed_type3_parquet_shuffled",
  "probe_id_dir": "processed_dataset/probe_ids_type3.csv",
  "qced_data_table": "QCed_samples_type3.csv",
  "valid_ratio": 0.1,
  "n_hvg": 49156,
  "max_fi": 500000,
  "do_train": true,
  "load_model": null,
  "mask_ratio": 0.3,
  "GEPC": true,
  "dab_weight": 1.0,
  "pretraining_dataset_name": "CpGs_type3",
  "epochs": 100,
  "ecs_thres": 0.0,
  "lr": 0.001,
  "batch_size": 8,
  "layer_size": 128,
  "nlayers": 6,
  "nhead": 4,
  "dropout": 0.1,
  "schedule_ratio": 0.9,
  "save_eval_interval": 10,
  "log_interval": 1000,
  "fast_transformer": true,
  "pre_norm": false,
  "amp": true
}


### Inspect the model weights

In [16]:
import torch
sd = torch.load(MODEL_WEIGHTS, map_location="cpu")
print("Type:", type(sd))
if isinstance(sd, dict):
    keys = list(sd.keys())
    print("Number of keys:", len(keys))
    print("First 15 keys:")
    for k in keys[:15]:
        print("  ", k)

Type: <class 'collections.OrderedDict'>
Number of keys: 100
First 15 keys:
   encoder.embedding.weight
   encoder.enc_norm.weight
   encoder.enc_norm.bias
   value_encoder.linear1.weight
   value_encoder.linear1.bias
   value_encoder.linear2.weight
   value_encoder.linear2.bias
   value_encoder.norm.weight
   value_encoder.norm.bias
   transformer_encoder.layers.0.self_attn.Wqkv.weight
   transformer_encoder.layers.0.self_attn.Wqkv.bias
   transformer_encoder.layers.0.self_attn.out_proj.weight
   transformer_encoder.layers.0.self_attn.out_proj.bias
   transformer_encoder.layers.0.linear1.weight
   transformer_encoder.layers.0.linear1.bias


### Inspect the methylation data + vocab


In [19]:
import json, anndata as ad

adata = ad.read_h5ad(INPUT_H5AD)
print("\nData shape (patients, probes):", adata.shape)
print("obs columns:", list(adata.obs.columns))
print("first 3 probe names:", list(adata.var_names[:3]))

with open(VOCAB_PATH) as f:
    vocab = json.load(f)
print("\nvocab type:", type(vocab), "| len:", len(vocab))
if isinstance(vocab, dict):
    print("lowest-id tokens:", sorted(vocab.items(), key=lambda kv: kv[1])[:6])
else:
    print("first 6 entries:", vocab[:6])


Data shape (patients, probes): (800, 49156)
obs columns: ['case_id', 'cancer_type', 'file_id']
first 3 probe names: ['0', '1', '2']

vocab type: <class 'dict'> | len: 49159
lowest-id tokens: [('<pad>', 0), ('<cls>', 1), ('<eoc>', 2), ('cg00000109', 3), ('cg00000292', 4), ('cg00002033', 5)]


### Define the model (rebuilt skeleton)

In [20]:
import torch, torch.nn as nn, torch.nn.functional as F

class SelfAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.n_heads, self.head_dim = n_heads, d_model // n_heads
        self.Wqkv = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.Wqkv(x).split(C, dim=-1)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        out = F.scaled_dot_product_attention(q, k, v)   # memory-efficient
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(out)

class TransformerLayer(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.self_attn = SelfAttention(d_model, n_heads)
        self.linear1, self.linear2 = nn.Linear(d_model, d_model), nn.Linear(d_model, d_model)
        self.norm1, self.norm2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
    def forward(self, x):
        x = self.norm1(x + self.self_attn(x))
        x = self.norm2(x + self.linear2(F.gelu(self.linear1(x))))
        return x

class TransformerStack(nn.Module):
    def __init__(self, d_model, n_heads, n_layers):
        super().__init__()
        self.layers = nn.ModuleList([TransformerLayer(d_model, n_heads) for _ in range(n_layers)])
    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

class ProbeEncoder(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.enc_norm = nn.LayerNorm(d_model)
    def forward(self, ids):
        return self.enc_norm(self.embedding(ids))

class ValueEncoder(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.linear1, self.linear2 = nn.Linear(1, d_model), nn.Linear(d_model, d_model)
        self.norm = nn.LayerNorm(d_model)
    def forward(self, v):
        x = F.relu(self.linear1(v.unsqueeze(-1).float()))
        x = self.linear2(x)
        return self.norm(x)

class MethylGPT(nn.Module):
    def __init__(self, vocab_size, d_model=128, n_heads=4, n_layers=6):
        super().__init__()
        self.encoder = ProbeEncoder(vocab_size, d_model)
        self.value_encoder = ValueEncoder(d_model)
        self.transformer_encoder = TransformerStack(d_model, n_heads, n_layers)
    def forward(self, gene_ids, values):
        x = self.encoder(gene_ids) + self.value_encoder(values)
        x = self.transformer_encoder(x)
        return x[:, 0, :]   # CLS position = patient embedding

print("Model defined.")

Model defined.


### Load the pretrained weights

In [21]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
sd = torch.load(MODEL_WEIGHTS, map_location='cpu')

vocab_size, d_model = sd['encoder.embedding.weight'].shape
n_layers = 1 + max(int(k.split('.')[2]) for k in sd if k.startswith('transformer_encoder.layers.'))
print("vocab_size:", vocab_size, "| d_model:", d_model, "| n_layers:", n_layers)

model = MethylGPT(vocab_size=vocab_size, d_model=d_model, n_heads=4, n_layers=n_layers)
model_state = {k: v for k, v in sd.items()
               if k.startswith(('encoder.', 'value_encoder.', 'transformer_encoder.'))}
missing, unexpected = model.load_state_dict(model_state, strict=False)
print("Missing keys   :", missing)
print("Unexpected keys:", unexpected)
model.to(device).eval()
print("Loaded.")

vocab_size: 49159 | d_model: 128 | n_layers: 6
Missing keys   : []
Unexpected keys: []
Loaded.


### Build inputs — token IDs + values

In [22]:
import numpy as np

X = adata.X if isinstance(adata.X, np.ndarray) else adata.X.toarray()
X = X.astype(np.float32) # (800, 49156) beta values
n_samples, n_probes = X.shape
print("Samples:", n_samples, "| Probes:", n_probes)

CLS_ID, N_SPECIAL = 1, 3
probe_ids = np.arange(n_probes) + N_SPECIAL # ids 3 .. 49158
gene_ids_template = torch.tensor([CLS_ID] + probe_ids.tolist(), dtype=torch.long)  # (49157,)
CLS_VALUE = -2.0 # constant value placed at the <cls> slot
print("Sequence length (cls + probes):", gene_ids_template.shape[0])

Samples: 800 | Probes: 49156
Sequence length (cls + probes): 49157


### Extract embeddings

In [23]:
from tqdm import tqdm

try:
    from torch.nn.attention import sdpa_kernel, SDPBackend
    attn_ctx = lambda: sdpa_kernel([SDPBackend.EFFICIENT_ATTENTION, SDPBackend.FLASH_ATTENTION])
except Exception:
    attn_ctx = lambda: torch.backends.cuda.sdp_kernel(
        enable_flash=True, enable_math=False, enable_mem_efficient=True)

gene_ids = gene_ids_template.unsqueeze(0).to(device) # (1, 49157)
embeddings = []

with torch.no_grad():
    for i in tqdm(range(n_samples)):
        values = np.concatenate([[CLS_VALUE], X[i]]) # (49157,)
        values_t = torch.tensor(values, dtype=torch.float32).unsqueeze(0).to(device)
        with attn_ctx():
            emb = model(gene_ids, values_t) # (1, 128)
        embeddings.append(emb.squeeze(0).cpu().numpy())
        if (i + 1) % 100 == 0:
            np.save(OUTPUT_EMBEDDINGS, np.array(embeddings))

embeddings = np.array(embeddings)
print("Done. Shape:", embeddings.shape)

100%|██████████| 800/800 [53:45<00:00,  4.03s/it]

Done. Shape: (800, 128)


 ### Save output

In [24]:
import numpy as np

np.save(OUTPUT_EMBEDDINGS, embeddings)

print("Saved to:", OUTPUT_EMBEDDINGS)
print("Shape:", embeddings.shape)
print("Any NaN:", np.isnan(embeddings).any())

Saved to: /content/drive/MyDrive/multiomics-project/data/processed/methylgpt_embeddings.npy
Shape: (800, 128)
Any NaN: False


In [25]:
emb = np.load(OUTPUT_EMBEDDINGS)

print("=== Final Result ===")
print(f"Total patients : {emb.shape[0]}")
print(f"Embedding size : {emb.shape[1]}")
print()

print("=== Samples per cancer type ===")
counts = adata.obs['cancer_type'].value_counts()
for cancer, count in counts.items():
    print(f"  {cancer:15s}  {count} patients")
print(f"  {'TOTAL':15s}  {counts.sum()} patients")
print()

print("=== One embedding per cancer type (first 5 values) ===")
for cancer in counts.index:
    mask = adata.obs['cancer_type'] == cancer
    idx = mask.values.argmax()
    preview = emb[idx, :5].round(3)
    print(f"  {cancer:15s}  {preview}")

print()
print("No NaN:", not np.isnan(emb).any())
print("No Inf:", not np.isinf(emb).any())

=== Final Result ===
Total patients : 800
Embedding size : 128

=== Samples per cancer type ===
  TCGA-BRCA        134 patients
  TCGA-LUAD        134 patients
  TCGA-COAD        133 patients
  TCGA-KIRC        133 patients
  TCGA-LIHC        133 patients
  TCGA-THCA        133 patients
  TOTAL            800 patients

=== One embedding per cancer type (first 5 values) ===
  TCGA-BRCA        [-0.021  0.031  0.082  0.361  0.019]
  TCGA-LUAD        [-0.021  0.031  0.082  0.361  0.019]
  TCGA-COAD        [-0.021  0.031  0.082  0.361  0.019]
  TCGA-KIRC        [-0.021  0.031  0.082  0.361  0.019]
  TCGA-LIHC        [-0.021  0.031  0.082  0.361  0.019]
  TCGA-THCA        [-0.021  0.031  0.082  0.361  0.019]

No NaN: True
No Inf: True
